[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/TunaLee/posco/blob/main/notebooks/day13_solution.ipynb)

# Day 13 · 정답 — Docker 다루기

Codex 에게 시킬 것을 정하고, 받은 파일을 읽고 고친다

---

### 시작하기 전에

1. **파일 → 드라이브에 사본 저장** 을 먼저 누른다. 안 하면 고친 내용이 남지 않는다.
2. 셀을 고르고 **Shift + Enter** 로 실행한다.

`live` 와 `lab` 의 모든 문제에 대한 정답본이다.
수강생은 먼저 스스로 풀어 본 뒤에 연다.

두 벌을 합쳐 담으므로 **문제 번호가 `lab` 과 다르다.** 번호 대신
**지문으로 찾는다.**

문제는 실행하면 `assert` 로 자가 채점된다. 맞으면 `통과` 가 찍히고,
틀리면 기대값과 실제값이 같이 나온다.

## 1. 이 노트북이 하는 일

In [ ]:
# 검사에 쓸 도구만 준비한다. 설치할 것이 없다.
import re, textwrap

def 줄번호(글, 찾을것):
    for i, line in enumerate(글.strip().split('\n'), 1):
        if 찾을것 in line:
            return i
    return None

print('준비됐다')

## 2. 여섯 칸 적기

### 조별로 풀기

2~3명이 한 조로 상의하며 푼다.

> **실습문제 1.** 담을 앱을 하나 골라 **여섯 칸**을 채운다.
> 2~3명이 한 조로 상의한다. 앞 회차에서 만든 것 중에 고른다.
> 모르는 칸은 비워 두지 말고 「확인 필요」라고 적는다.

In [ ]:
여섯칸 = {
    '언어와 버전':  '파이썬 3.12',
    '라이브러리':   'requirements.txt 에 있는 것 그대로',
    '시작 명령':    'python app.py',
    '여는 포트':    '8000',
    '데이터':      'cell_process.csv — 이미지에 넣지 않고 밖에서 물린다',
    '키':         'NVIDIA_API_KEY — 값은 주지 않는다',
}
for k, v in 여섯칸.items():
    print('%-10s %s' % (k, v))
assert '___' not in str(여섯칸), '여섯 칸을 다 채운다'
print()
print('빈 칸 없음. 다음 문제에서 이것을 프롬프트로 옮긴다.')

## 3. 프롬프트로 옮기기

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 2.** 여섯 칸을 프롬프트로 만든다. **빠뜨리면 안 되는 두 줄**이 뒤에 있다.
> 그 두 줄이 없으면 Codex 가 늘 같은 실수를 한다.

In [ ]:
# 여섯칸 을 그대로 쓴다
프롬프트 = f'''Dockerfile 을 만들어 줘.

{여섯칸['언어와 버전']} 을 쓴다. 바탕 이미지는 slim 판으로.
꾸러미는 {여섯칸['라이브러리']}
시작 명령은 {여섯칸['시작 명령']}
{여섯칸['여는 포트']} 포트를 연다

앱은 127.0.0.1 이 아니라 0.0.0.0 으로 열게 한다
키는 환경변수로 받기만 한다. 값은 적지 마라
'''
print(프롬프트)

assert '0.0.0.0' in 프롬프트, '여는 주소를 못 박아야 한다'
assert '값은 적지' in 프롬프트 or '값을 적지' in 프롬프트, '키 값을 적지 말라고 해야 한다'
print()
print('두 줄이 들어갔다. 이 프롬프트를 그대로 Codex 에 붙여 넣는다.')

## 4. 틀린 Dockerfile 찾기

In [ ]:
# 검토할 Dockerfile 셋. 일부러 틀린 곳을 넣어 두었다.
후보 = [
    '바탕 이미지에 버전을 안 적었다',
    '키 값을 이미지에 박아 넣었다',
    '앱이 127.0.0.1 로 열려 밖에서 못 붙는다',
    '폴더를 통째로 복사해 잡동사니가 들어간다',
    '꾸러미를 코드보다 나중에 깔아 빌드가 느리다',
    '데이터를 이미지에 넣어 상자를 지우면 사라진다',
    '틀린 곳이 없다',
]
for i, x in enumerate(후보, 1):
    print('%d. %s' % (i, x))

In [ ]:
D1 = '''
FROM python:latest
WORKDIR /app
COPY . .
RUN pip install -r requirements.txt
ENV NVIDIA_API_KEY=nvapi-abc123
EXPOSE 8000
CMD ["python", "app.py"]
'''
print(D1)

In [ ]:
D2 = '''
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py .
COPY cell_process.csv .
EXPOSE 8000
CMD ["python", "app.py"]
'''
print(D2)

### 같이 풀기

수업 중에 같이 푼다.

> **실습문제 3.** 위 `D1` 에서 틀린 곳을 **넷** 고른다. 번호로 적는다.
> 후보 목록에서 고른다. 순서는 상관없다.

In [ ]:
# 후보 번호를 넣는다
답1 = [1, 2, 4, 5]

정답 = {1, 2, 4, 5}
assert set(답1) == 정답, '다시 본다. 힌트 — latest · ENV · COPY . . · pip 순서'
for n in sorted(답1):
    print('%d. %s' % (n, 후보[n-1]))
print()
print('넷 다 찾았다. 이 넷이 실제로 제일 자주 나온다.')

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 1.** `D2` 는 앞의 것보다 낫다. 그래도 **하나**가 남았다.
> 순서도 맞고 키도 없다. 무엇이 걸리나.

In [ ]:
# 후보 번호 하나
답2 = [6]

assert set(답2) == {6}, '데이터를 어디에 두었나 다시 본다'
print(후보[답2[0]-1])
print()
print('CSV 를 이미지에 넣었다. 데이터가 바뀔 때마다 이미지를 다시 만들어야 하고,')
print('상자를 지우면 안에 쌓인 것이 같이 사라진다. 밖에서 물려야 한다.')

## 5. compose.yml 읽기

In [ ]:
C1 = '''
services:
  web:
    build: ./web
    ports: ["3000:3000"]
    environment:
      API_URL: http://api:8000

  api:
    build: ./api
    ports: ["8000:8000"]

  db:
    image: postgres:16
    ports: ["5432:5432"]
    environment:
      POSTGRES_PASSWORD: mypassword
'''
print(C1)

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 2.** `C1` 에서 **밖으로 열지 않아야 하는 서비스**를 고르고, 그 까닭을 한 줄 적는다.
> `web` 은 사람이 브라우저로 쓴다. 나머지 둘은 누가 쓰나.

In [ ]:
# 서비스 이름과 까닭
닫아야할것 = 'db'
까닭 = '사람이 직접 쓰지 않는다. web 과 api 만 부르면 되니 안에서만 열면 된다'
print(닫아야할것, '—', 까닭)

assert 닫아야할것 == 'db', 'ports 를 열 이유가 없는 것을 고른다'
assert '___' not in 까닭, '까닭을 적는다'
print()
print('ports 를 지우면 같은 compose 안의 web · api 만 붙는다.')
print('한 가지 더 — POSTGRES_PASSWORD 를 파일에 그대로 적었다. .env 로 빼야 한다.')

## 6. 폐쇄망용으로 고치기

### 스스로 풀기

각자 푼다. 막히면 손을 든다.

> **빈칸 문제 3.** `C1` 을 폐쇄망용으로 고친다. **`build:` 를 `image:` 로** 바꾼다.
> 이미지 이름은 영문 소문자로 짓는다. 한글은 빌드에서 거절당한다.

In [ ]:
C2 = C1
C2 = C2.replace('build: ./web', 'image: posco-web:1.0')
C2 = C2.replace('build: ./api', 'image: posco-api:1.0')
print(C2)

assert 'build:' not in C2, 'build 가 남아 있으면 받는 쪽이 다시 만들려 한다'
assert '___' not in C2, '이미지 이름을 채운다'
import re
for name in re.findall(r'image: (\S+)', C2):
    assert re.match(r'^[a-z0-9._/-]+(:[a-zA-Z0-9._-]+)?$', name), '이름은 영문 소문자로: ' + name
print()
print('build 가 없어졌다. 이제 load 한 이미지로만 뜬다.')

## 7. 터미널에서 할 것

### 조별로 풀기

2~3명이 한 조로 상의하며 푼다.

> **실습문제 4.** **옮길 것 목록**을 만든다. 다른 조 PC 에서 띄우려면 무엇을 넘겨야 하나.
> 넘기지 말아야 할 것도 같이 적는다.

In [ ]:
넘길것 = ['images.tar', 'compose.yml', 'env.example']
안넘길것 = ['.env', 'node_modules 나 venv 폴더']
print('넘긴다   :', ' · '.join(넘길것))
print('안 넘긴다 :', ' · '.join(안넘길것))
assert '___' not in str(넘길것) + str(안넘길것), '다 채운다'
assert any('.env' in x for x in 안넘길것), '키가 든 파일은 안 넘긴다'
print()
print('소스 코드도 안 넘겨도 된다. 이미지 안에 이미 들어 있다.')